Import the core libraries used throughout the notebook: `pandas` for data handling, `matplotlib`/`seaborn` for plotting, `numpy` for numerical operations, and `requests` to test the data URLs.

In [56]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import requests
%matplotlib inline

## Setup and Data Loading

This section imports the libraries needed for the analysis, briefly tests that the data source is reachable, then loads all four datasets from GitHub into pandas dataframes for analysis.

Load all four raw datasets directly from the GitHub repository using their raw file URLs, so anyone on the team can run this notebook without changing local file paths. Each dataset is read into its own dataframe: client demographics, the two parts of the web activity log, and the experiment client roster.


In [57]:
df_final_demo = "https://raw.githubusercontent.com/winifredpaul/vanguard-ui-experiment-analysis/main/data/df_final_demo.txt"
df_final_web_data_pt_1 = "https://raw.githubusercontent.com/winifredpaul/vanguard-ui-experiment-analysis/main/data/df_final_web_data_pt_1.zip"
df_final_web_data_pt_2 = "https://raw.githubusercontent.com/winifredpaul/vanguard-ui-experiment-analysis/main/data/df_final_web_data_pt_2.zip"
df_final_experiment_clients = "https://raw.githubusercontent.com/winifredpaul/vanguard-ui-experiment-analysis/main/data/df_final_experiment_clients.txt"
clients_demo_df = pd.read_csv(df_final_demo)
web_data_pt1_df = pd.read_csv(df_final_web_data_pt_1, compression="zip")
web_data_pt2_df = pd.read_csv(df_final_web_data_pt_2, compression="zip")
experiment_clients_df = pd.read_csv(df_final_experiment_clients)

## Dataframe Reference

| Dataframe | Description |
|---|---|
| `clients_demo_df` | Raw client demographics — one row per client, including tenure, age, gender, account count, balance, and recent call/logon activity. |
| `web_data_pt1_df` | First half of the raw web activity log — one row per client interaction with the digital process (page views/steps). |
| `web_data_pt2_df` | Second half of the raw web activity log, sharing the same structure as `web_data_pt1_df`. |
| `experiment_clients_df` | Roster of clients and their assigned experiment group (`Test`, `Control`, or missing if not part of the experiment). |
| `clients_demo_exp_df` | `clients_demo_df` merged with `experiment_clients_df`, adding the `Variation` column and a `part_of_experiment` flag (`Yes`/`No`) for every client. |
| `experiment_only_df` | `clients_demo_exp_df` filtered down to only clients actually assigned to Test or Control (excludes non-participants). |
| `experiment_final_df` | `experiment_only_df` with rows containing missing values dropped — this is the **analysis-ready** client-level dataset for Phase 2/3 (KPIs, hypothesis testing). |
| `combined_web_data_df` | `web_data_pt1_df` and `web_data_pt2_df` combined (stacked) into a single web activity log, with duplicates removed and `date_time` converted to a proper datetime type. |
| `client_last_step` | One row per client, showing their most recently recorded process step and experiment group — built from `combined_web_data_df`. *(See the note on this dataframe below — it summarizes each client's last event overall, not per visit.)* |

**Note:** `client_last_step` currently identifies each client's furthest step based on their **most recent activity overall**, which may combine multiple separate visits into one summary row. For metrics like completion rate, we may want to revisit this at the **visit level** (`visit_id`) instead of the client level — see the "Spot-Checking Individual Client Histories" section for an example of why this matters.

## Analysing "df_final_demo" dataframe

This section performs an initial inspection of the client demographics dataset: checking data types and non-null counts, reviewing the unique values and distribution of the `gendr` column, checking for missing values and duplicate rows, reviewing summary statistics for the numeric columns, and isolating any rows with missing data for closer inspection.

In [58]:
# Check data types and non-null counts for each column
clients_demo_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70609 entries, 0 to 70608
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   client_id         70609 non-null  int64  
 1   clnt_tenure_yr    70595 non-null  float64
 2   clnt_tenure_mnth  70595 non-null  float64
 3   clnt_age          70594 non-null  float64
 4   gendr             70595 non-null  object 
 5   num_accts         70595 non-null  float64
 6   bal               70595 non-null  float64
 7   calls_6_mnth      70595 non-null  float64
 8   logons_6_mnth     70595 non-null  float64
dtypes: float64(7), int64(1), object(1)
memory usage: 4.8+ MB


In [59]:
# Count missing values in each column
clients_demo_df.isnull().sum()

client_id            0
clnt_tenure_yr      14
clnt_tenure_mnth    14
clnt_age            15
gendr               14
num_accts           14
bal                 14
calls_6_mnth        14
logons_6_mnth       14
dtype: int64

In [60]:
# Isolate rows that have any missing values, for closer inspection
clients_demo_df[clients_demo_df.isnull().any(axis=1)]

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth
4164,7402828,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8316,355337,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8677,8412164,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9583,4666211,8.0,106.0,NaN,F,2.0,42550.55,4.0,7.0
13444,2222915,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18066,4876926,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25961,5277910,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28432,7616759,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35323,8191345,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
43518,1227228,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [61]:
# Check for fully duplicated rows (all columns identical)
clients_demo_df.duplicated().sum()

np.int64(0)

In [62]:
# Check for duplicate client IDs specifically (different from full-row duplicates)
clients_demo_df["client_id"].duplicated().sum()

np.int64(0)

In [63]:
# Check the distinct values in the gendr column
clients_demo_df["gendr"].unique()

array(['U', 'M', 'F', nan, 'X'], dtype=object)

In [64]:
# Check how many clients fall into each gender category
clients_demo_df["gendr"].value_counts()

gendr
U    24122
M    23724
F    22746
X        3
Name: count, dtype: int64

In [65]:
# Review summary statistics (mean, min, max, quartiles) for the numeric columns
clients_demo_df.describe()

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,num_accts,bal,calls_6_mnth,logons_6_mnth
count,7.060900e+04,70595.000000,70595.000000,70594.000000,70595.000000,7.059500e+04,70595.000000,70595.000000
mean,5.004992e+06,12.052950,150.659367,46.442240,2.255528,1.474452e+05,3.382478,5.566740
std,2.877278e+06,6.871819,82.089854,15.591273,0.534997,3.015087e+05,2.236580,2.353286
min,1.690000e+02,2.000000,33.000000,13.500000,1.000000,1.378942e+04,0.000000,1.000000
25%,2.519329e+06,6.000000,82.000000,32.500000,2.000000,3.734683e+04,1.000000,4.000000
50%,5.016978e+06,11.000000,136.000000,47.000000,2.000000,6.333290e+04,3.000000,5.000000
75%,7.483085e+06,16.000000,192.000000,59.000000,2.000000,1.375449e+05,6.000000,7.000000
max,9.999839e+06,62.000000,749.000000,96.000000,8.000000,1.632004e+07,7.000000,9.000000


In [66]:
# Look at the youngest clients to check the age data makes sense
clients_demo_df.sort_values("clnt_age").head()

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth
70216,5417612,4.0,59.0,13.5,U,2.0,24435.08,7.0,7.0
67700,6250331,4.0,59.0,14.0,U,2.0,19945.35,3.0,3.0
58281,3677152,22.0,273.0,14.0,F,2.0,16989.14,1.0,1.0
51837,4226013,5.0,65.0,14.5,U,3.0,32835.33,5.0,5.0
63598,49451,14.0,178.0,14.5,M,2.0,15487.91,2.0,2.0


`clnt_tenure_yr` is a rounded-down (floored) version of `clnt_tenure_mnth ÷ 12`, not an independently recorded value — confirmed by comparing `clnt_tenure_mnth // 12` against `clnt_tenure_yr`, which match exactly. This isn't a data quality issue; it just means `clnt_tenure_mnth` is the more precise field to use for any tenure-based calculations going forward.

In [67]:
# Check if clnt_tenure_yr = floor(clnt_tenure_mnth / 12)
clients_demo_df["tenure_check_correct"] = (clients_demo_df["clnt_tenure_mnth"] // 12)

clients_demo_df[["clnt_tenure_yr", "clnt_tenure_mnth", "tenure_check_correct"]].head(10)

,clnt_tenure_yr,clnt_tenure_mnth,tenure_check_correct
0,6.0,73.0,6.0
1,7.0,94.0,7.0
2,5.0,64.0,5.0
3,16.0,198.0,16.0
4,12.0,145.0,12.0
5,5.0,71.0,5.0
6,5.0,66.0,5.0
7,30.0,361.0,30.0
8,30.0,369.0,30.0
9,15.0,189.0,15.0


## Analysing "df_final_web_data_pt_1" dataframe

This section performs an initial inspection of the first web activity file: checking data types and non-null counts, missing values, and duplicate rows, previewing the first few rows to understand the structure of the data, confirming the distinct process steps match the expected funnel stages (start, step_1, step_2, step_3, confirm), checking how many unique visitors and sessions are captured, and reviewing the date range covered — establishing a baseline to sanity-check against once this file is combined with web_data_pt2_df.

In [68]:
# Check data types and non-null counts for the first web activity file
web_data_pt1_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 343141 entries, 0 to 343140
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   client_id     343141 non-null  int64 
 1   visitor_id    343141 non-null  object
 2   visit_id      343141 non-null  object
 3   process_step  343141 non-null  object
 4   date_time     343141 non-null  object
dtypes: int64(1), object(4)
memory usage: 13.1+ MB


In [69]:
# Check for missing values
web_data_pt1_df.isnull().sum()

client_id       0
visitor_id      0
visit_id        0
process_step    0
date_time       0
dtype: int64

In [70]:
# Check for fully duplicated rows
web_data_pt1_df.duplicated().sum()

np.int64(2095)

In [71]:
# Preview the first few rows to understand the structure of the web activity log
web_data_pt1_df.head()

,client_id,visitor_id,visit_id,process_step,date_time
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04


In [72]:
# Check the distinct process steps to confirm the expected funnel stages
web_data_pt1_df["process_step"].unique()

array(['step_3', 'step_2', 'step_1', 'start', 'confirm'], dtype=object)

In [73]:
# Check how many unique visitors and sessions are captured in this file
print("Unique visitors:", web_data_pt1_df["visitor_id"].nunique())
print("Unique visits:", web_data_pt1_df["visit_id"].nunique())

Unique visitors: 62936
Unique visits: 75256


In [74]:
# Check the date range covered by this file
web_data_pt1_df["date_time"] = pd.to_datetime(web_data_pt1_df["date_time"])
print("First activity:", web_data_pt1_df["date_time"].min())
print("Last activity:", web_data_pt1_df["date_time"].max())

First activity: 2017-03-15 00:03:03
Last activity: 2017-04-30 23:59:16


## Analysing "df_final_web_data_pt_2" dataframe

This section performs an initial inspection of the second web activity file: checking data types and non-null counts, missing values, and duplicate rows, previewing the first few rows to confirm it matches pt_1's structure, confirming the distinct process steps align with the same funnel stages (start, step_1, step_2, step_3, confirm), checking how many unique visitors and sessions are captured, and reviewing the date range covered. Finally, the columns of both files are compared directly to confirm they match exactly before combining them with pd.concat().

In [75]:
# Check data types and non-null counts for the second web activity file
web_data_pt2_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 412264 entries, 0 to 412263
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   client_id     412264 non-null  int64 
 1   visitor_id    412264 non-null  object
 2   visit_id      412264 non-null  object
 3   process_step  412264 non-null  object
 4   date_time     412264 non-null  object
dtypes: int64(1), object(4)
memory usage: 15.7+ MB


In [76]:
# Check for missing values
web_data_pt2_df.isnull().sum()

client_id       0
visitor_id      0
visit_id        0
process_step    0
date_time       0
dtype: int64

In [77]:
# Check for fully duplicated rows
web_data_pt2_df.duplicated().sum()

np.int64(8669)

In [78]:
# Preview the first few rows to confirm this file matches pt_1's structure
web_data_pt2_df.head()

,client_id,visitor_id,visit_id,process_step,date_time
0,763412,601952081_10457207388,397475557_40440946728_419634,confirm,2017-06-06 08:56:00
1,6019349,442094451_91531546617,154620534_35331068705_522317,confirm,2017-06-01 11:59:27
2,6019349,442094451_91531546617,154620534_35331068705_522317,step_3,2017-06-01 11:58:48
3,6019349,442094451_91531546617,154620534_35331068705_522317,step_2,2017-06-01 11:58:08
4,6019349,442094451_91531546617,154620534_35331068705_522317,step_1,2017-06-01 11:57:58


In [79]:
# Check the distinct process steps match pt_1's funnel stages
web_data_pt2_df["process_step"].unique()

array(['confirm', 'step_3', 'step_2', 'step_1', 'start'], dtype=object)

In [80]:
# Check how many unique visitors and sessions are captured in this file
print("Unique visitors:", web_data_pt2_df["visitor_id"].nunique())
print("Unique visits:", web_data_pt2_df["visit_id"].nunique())

Unique visitors: 71042
Unique visits: 82841


In [81]:
# Check the date range covered by this file
web_data_pt2_df["date_time"] = pd.to_datetime(web_data_pt2_df["date_time"])
print("First activity:", web_data_pt2_df["date_time"].min())
print("Last activity:", web_data_pt2_df["date_time"].max())

First activity: 2017-05-01 00:00:26
Last activity: 2017-06-20 23:59:57


In [82]:
# Confirm both web data files have identical columns, in the same order,
# before combining them with pd.concat()
list(web_data_pt1_df.columns) == list(web_data_pt2_df.columns)

True

## Analysing "df_final_experiment_clients" dataframe

This section performs an initial inspection of the experiment roster: checking data types and non-null counts, missing values in the Variation column (clients with no assignment weren't part of the experiment), full-row and duplicate client_id checks to confirm each client appears only once, the distinct values of Variation (Test, Control, or missing), and the count of clients in each group.

In [83]:
# Check data types and non-null counts for the experiment roster
experiment_clients_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70609 entries, 0 to 70608
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   client_id  70609 non-null  int64 
 1   Variation  50500 non-null  object
dtypes: int64(1), object(1)
memory usage: 1.1+ MB


In [84]:
# Check for missing Variation values (clients not part of the experiment)
experiment_clients_df.isnull().sum()

client_id        0
Variation    20109
dtype: int64

In [85]:
# Check for fully duplicated rows
experiment_clients_df.duplicated().sum()

np.int64(0)

In [86]:
# Check for duplicate client IDs in the roster
experiment_clients_df["client_id"].duplicated().sum()

np.int64(0)

In [87]:
# Confirm the only possible values are Test, Control, or missing
experiment_clients_df["Variation"].unique()

array(['Test', 'Control', nan], dtype=object)

In [88]:
# Check how many clients are in each experiment group
experiment_clients_df["Variation"].value_counts(dropna=False)

Variation
Test       26968
Control    23532
NaN        20109
Name: count, dtype: int64

## Merging Client Demographic file with Clients who were part of experiment file

This step checks how many clients overlap between the two datasets before merging them.

- First, it counts the total number of unique clients in the demographics file (`clients_demo_df`).
- Then, it counts how many clients were actually assigned to a Test or Control group in the experiment file (ignoring clients with no assignment, i.e. `Variation` is blank).
- Next, it merges the two files together on `client_id`, keeping only clients that appear in both datasets (an "inner" merge).
- From that merged result, it counts how many of those overlapping clients also have a valid experiment assignment (Test or Control).
- Finally, it prints all three numbers side by side, so we can sanity-check that the overlap makes sense before doing the real merge in the next step.

This is a validation check, not the actual merge used later — its purpose is to confirm the two datasets line up correctly (e.g. that most/all demo clients also appear in the experiment file) before we commit to merging them for real.

In [89]:
# STEP 1: Count unique clients in dataset 1
dataset_1_clients = clients_demo_df["client_id"].nunique()

# STEP 2: Count unique clients assigned to the experiment (exclude NaN)
dataset_4_clients = (
    experiment_clients_df
    .loc[experiment_clients_df["Variation"].notna(), "client_id"]
    .nunique()
)

# STEP 3: Find clients appearing in both datasets
overlapping_clients = clients_demo_df.merge(
    experiment_clients_df,
    on="client_id",
    how="inner"
)

# STEP 4: Count unique overlapping clients who participated in the experiment
overlap_count = (
    overlapping_clients
    .loc[overlapping_clients["Variation"].notna(), "client_id"]
    .nunique()
)

# STEP 5: Display the results
print("Unique clients in dataset 1:", dataset_1_clients)
print("Unique experiment clients:", dataset_4_clients)
print("Clients appearing in both datasets:", overlap_count)

# STEP 6: Preview the overlapping clients
overlapping_clients.head(10)

Unique clients in dataset 1: 70609
Unique experiment clients: 50500
Clients appearing in both datasets: 50500


,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,tenure_check_correct,Variation
0,836976,6.0,73.0,60.5,U,2.0,45105.30,6.0,9.0,6.0,Test
1,2304905,7.0,94.0,58.0,U,2.0,110860.30,6.0,9.0,7.0,Control
2,1439522,5.0,64.0,32.0,U,2.0,52467.79,6.0,9.0,5.0,Test
3,1562045,16.0,198.0,49.0,M,2.0,67454.65,3.0,6.0,16.0,Test
4,5126305,12.0,145.0,33.0,F,2.0,103671.75,0.0,3.0,12.0,Control
5,3727881,5.0,71.0,30.5,U,2.0,23915.60,0.0,3.0,5.0,Control
6,272934,5.0,66.0,58.5,U,2.0,27021.42,2.0,5.0,5.0,Control
7,388801,30.0,361.0,57.5,M,5.0,522498.72,1.0,4.0,30.0,Test
8,285619,30.0,369.0,67.5,M,2.0,299388.72,3.0,6.0,30.0,Control
9,8198645,15.0,189.0,54.5,F,2.0,382303.83,6.0,9.0,15.0,Test


Now that the overlap has been validated, this step performs the real merge: it adds the `Variation` column from the experiment roster onto the client demographics dataset, using a left join so that every client from the demographics file is kept, whether or not they were part of the experiment.

A new column, `part_of_experiment`, is then created from `Variation`: it's marked `"Yes"` if the client has a Test/Control assignment, and `"No"` if `Variation` is blank (meaning they weren't included in the experiment). This flag makes it easy to filter down to experiment-only clients in later steps.

In [90]:
# Merge the Variation column from the experiment roster into the demo dataset
# Using a left join keeps every client, whether or not they were part of the experiment
clients_demo_exp_df = clients_demo_df.merge(
    experiment_clients_df[["client_id", "Variation"]],
    on="client_id",
    how="left"
)

# Flag whether each client was part of the experiment
clients_demo_exp_df["part_of_experiment"] = clients_demo_exp_df["Variation"].notna().map({
    True: "Yes",
    False: "No"
})

clients_demo_exp_df.head()

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,tenure_check_correct,Variation,part_of_experiment
0,836976,6.0,73.0,60.5,U,2.0,45105.30,6.0,9.0,6.0,Test,Yes
1,2304905,7.0,94.0,58.0,U,2.0,110860.30,6.0,9.0,7.0,Control,Yes
2,1439522,5.0,64.0,32.0,U,2.0,52467.79,6.0,9.0,5.0,Test,Yes
3,1562045,16.0,198.0,49.0,M,2.0,67454.65,3.0,6.0,16.0,Test,Yes
4,5126305,12.0,145.0,33.0,F,2.0,103671.75,0.0,3.0,12.0,Control,Yes


In [91]:
# Confirm structure and non-null counts after the merge
clients_demo_exp_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70609 entries, 0 to 70608
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   client_id             70609 non-null  int64  
 1   clnt_tenure_yr        70595 non-null  float64
 2   clnt_tenure_mnth      70595 non-null  float64
 3   clnt_age              70594 non-null  float64
 4   gendr                 70595 non-null  object 
 5   num_accts             70595 non-null  float64
 6   bal                   70595 non-null  float64
 7   calls_6_mnth          70595 non-null  float64
 8   logons_6_mnth         70595 non-null  float64
 9   tenure_check_correct  70595 non-null  float64
 10  Variation             50500 non-null  object 
 11  part_of_experiment    70609 non-null  object 
dtypes: float64(8), int64(1), object(3)
memory usage: 6.5+ MB


In [92]:
# Check how many clients fall into each group, including those not in the experiment
clients_demo_exp_df["Variation"].value_counts(dropna=False)

Variation
Test       26968
Control    23532
NaN        20109
Name: count, dtype: int64

In [93]:
# Check gender distribution across Test/Control/not-in-experiment groups
gender_by_variation = pd.crosstab(
    clients_demo_exp_df["Variation"],
    clients_demo_exp_df["gendr"]
)
print(gender_by_variation)

gendr         F     M     U  X
Variation                     
Control    7543  7970  8014  0
Test       8716  8977  9266  2


### Identifying Clients Participating in the A/B Experiment

To determine which clients were included in the A/B test, the client demographics dataset (clients_demo_df) was merged with the experiment roster (experiment_clients_df) using the client_id field.

The `Variation` column from the experiment dataset was added to the demographics dataset, indicating whether each client belonged to the **Test** group, **Control** group, or was **not part of the experiment** (`NaN`).

A new boolean column, `part_of_experiment`, was then created:

- **True** → Client participated in the A/B experiment (assigned to either the Test or Control group).
- **False** → Client was not included in the experiment.

This flag simplifies filtering the dataset for subsequent analyses, such as calculating performance metrics, comparing completion rates, and conducting hypothesis testing using only the experimental population.

## Combining the two web data files

In [94]:
# Stack the two web activity files into one (they share identical columns)
combined_web_data_df = pd.concat(
    [web_data_pt1_df, web_data_pt2_df],
    ignore_index=True
)

combined_web_data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 755405 entries, 0 to 755404
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   client_id     755405 non-null  int64         
 1   visitor_id    755405 non-null  object        
 2   visit_id      755405 non-null  object        
 3   process_step  755405 non-null  object        
 4   date_time     755405 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 28.8+ MB


In [95]:
# Check for duplicate rows introduced by combining the two files
combined_web_data_df.duplicated().sum()

np.int64(10764)

In [96]:
# Remove duplicate rows so events aren't double-counted downstream
combined_web_data_df = combined_web_data_df.drop_duplicates()

# Confirm no duplicates remain
combined_web_data_df.duplicated().sum()

np.int64(0)

In [97]:
# Confirm the combined file still only has the expected funnel stages
combined_web_data_df["process_step"].unique()

array(['step_3', 'step_2', 'step_1', 'start', 'confirm'], dtype=object)

In [98]:
# Convert date_time to a proper datetime type
combined_web_data_df["date_time"] = pd.to_datetime(combined_web_data_df["date_time"])

In [99]:
combined_web_data_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 744641 entries, 0 to 755404
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   client_id     744641 non-null  int64         
 1   visitor_id    744641 non-null  object        
 2   visit_id      744641 non-null  object        
 3   process_step  744641 non-null  object        
 4   date_time     744641 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 34.1+ MB


In [100]:
# Check the overall date range and confirm all activity falls within the experiment window
print("First activity:", combined_web_data_df["date_time"].min())
print("Last activity:", combined_web_data_df["date_time"].max())

experiment_start = pd.Timestamp("2017-03-15")
experiment_end = pd.Timestamp("2017-06-21")

outside_experiment = combined_web_data_df[
    (combined_web_data_df["date_time"] < experiment_start) | (combined_web_data_df["date_time"] > experiment_end)
]

print(f"\nActivities outside the experiment period: {len(outside_experiment)}")
outside_experiment.tail()

First activity: 2017-03-15 00:03:03
Last activity: 2017-06-20 23:59:57

Activities outside the experiment period: 0


,client_id,visitor_id,visit_id,process_step,date_time


In [101]:
# Spot-check one client's full activity history, sorted chronologically
client_id = 934

(
    combined_web_data_df[combined_web_data_df["client_id"] == client_id]
    .sort_values("date_time")
)

,client_id,visitor_id,visit_id,process_step,date_time
62623,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:36:30
62622,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:37:02
62621,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:38:24
62620,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:38:52


In [102]:
# Spot-check a client who completed the process (reached "confirm")
# Useful to confirm their last row really does show process_step = "confirm"
client_id = combined_web_data_df.loc[
    combined_web_data_df["process_step"] == "confirm", "client_id"
].iloc[0]

(
    combined_web_data_df[combined_web_data_df["client_id"] == client_id]
    .sort_values("date_time")
)

,client_id,visitor_id,visit_id,process_step,date_time
12,8320017,39393514_33118319366,960651974_70596002104_312201,start,2017-04-05 13:08:06
11,8320017,39393514_33118319366,960651974_70596002104_312201,step_1,2017-04-05 13:08:24
10,8320017,39393514_33118319366,960651974_70596002104_312201,step_2,2017-04-05 13:08:40
9,8320017,39393514_33118319366,960651974_70596002104_312201,step_3,2017-04-05 13:09:43
8,8320017,39393514_33118319366,960651974_70596002104_312201,confirm,2017-04-05 13:10:05


In [103]:
# Spot-check a client with multiple visits (visit_id changes),
# to see how groupby("client_id").last() behaves when someone has more than one session
multi_visit_clients = (
    combined_web_data_df.groupby("client_id")["visit_id"].nunique()
)
client_id = multi_visit_clients[multi_visit_clients > 1].index[0]

(
    combined_web_data_df[combined_web_data_df["client_id"] == client_id]
    .sort_values("date_time")
)

,client_id,visitor_id,visit_id,process_step,date_time
594421,805,831412807_82548325803,905546080_75813398358_250101,start,2017-06-08 01:10:29
594420,805,831412807_82548325803,905546080_75813398358_250101,step_1,2017-06-08 01:11:49
594431,805,831412807_82548325803,451173196_11661340552_563345,start,2017-06-15 19:09:13
594430,805,831412807_82548325803,451173196_11661340552_563345,step_1,2017-06-15 19:09:19
594429,805,831412807_82548325803,451173196_11661340552_563345,step_2,2017-06-15 19:09:26
594428,805,831412807_82548325803,451173196_11661340552_563345,step_2,2017-06-15 19:11:21
594427,805,831412807_82548325803,451173196_11661340552_563345,start,2017-06-15 19:11:28
594491,805,831412807_82548325803,72221519_52513154978_430287,start,2017-06-17 19:23:17
594490,805,831412807_82548325803,72221519_52513154978_430287,step_1,2017-06-17 19:23:20


## Determining the Furthest Step Reached per Client

Since clients can navigate back and forth between steps, we can't rely on their most recent chronological event to determine whether they completed the process — a client's very last recorded action might be an earlier step from a separate session. Instead, this section ranks each process step numerically and finds the highest rank each client ever reached, which more accurately reflects true completion.

The result is merged into `clients_demo_exp_df`, so completion status is available alongside each client's demographics and experiment group.

In [104]:
# Define the order of the process, so we can measure "furthest step reached"
step_order = {
    "start": 0,
    "step_1": 1,
    "step_2": 2,
    "step_3": 3,
    "confirm": 4
}

# Map each process_step to its numeric rank
combined_web_data_df["step_rank"] = combined_web_data_df["process_step"].map(step_order)

# For each client, find the highest step rank they ever reached
furthest_step = (
    combined_web_data_df
    .groupby("client_id")["step_rank"]
    .max()
    .reset_index()
)

# Map the numeric rank back to a readable step name
rank_to_step = {v: k for k, v in step_order.items()}
furthest_step["furthest_step_reached"] = furthest_step["step_rank"].map(rank_to_step)

furthest_step.head()

,client_id,step_rank,furthest_step_reached
0,169,4,confirm
1,336,0,start
2,546,4,confirm
3,555,4,confirm
4,647,4,confirm


In [105]:
# Merge the furthest step reached into the demographics + experiment dataframe
clients_demo_exp_df = clients_demo_exp_df.merge(
    furthest_step[["client_id", "furthest_step_reached"]],
    on="client_id",
    how="left"
)

clients_demo_exp_df.head()

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,tenure_check_correct,Variation,part_of_experiment,furthest_step_reached
0,836976,6.0,73.0,60.5,U,2.0,45105.30,6.0,9.0,6.0,Test,Yes,confirm
1,2304905,7.0,94.0,58.0,U,2.0,110860.30,6.0,9.0,7.0,Control,Yes,confirm
2,1439522,5.0,64.0,32.0,U,2.0,52467.79,6.0,9.0,5.0,Test,Yes,step_3
3,1562045,16.0,198.0,49.0,M,2.0,67454.65,3.0,6.0,16.0,Test,Yes,start
4,5126305,12.0,145.0,33.0,F,2.0,103671.75,0.0,3.0,12.0,Control,Yes,start


In [106]:
# Keep only clients who were actually assigned to Test or Control
experiment_only_df = clients_demo_exp_df[
    clients_demo_exp_df["Variation"].isin(["Test", "Control"])
].copy()

# Drop the participation flag — every row here is an experiment client by definition
experiment_only_df = experiment_only_df.drop(columns=["part_of_experiment"])

experiment_only_df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 50500 entries, 0 to 50499
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   client_id              50500 non-null  int64  
 1   clnt_tenure_yr         50488 non-null  float64
 2   clnt_tenure_mnth       50488 non-null  float64
 3   clnt_age               50487 non-null  float64
 4   gendr                  50488 non-null  object 
 5   num_accts              50488 non-null  float64
 6   bal                    50488 non-null  float64
 7   calls_6_mnth           50488 non-null  float64
 8   logons_6_mnth          50488 non-null  float64
 9   tenure_check_correct   50488 non-null  float64
 10  Variation              50500 non-null  object 
 11  furthest_step_reached  50500 non-null  object 
dtypes: float64(8), int64(1), object(3)
memory usage: 5.0+ MB


In [107]:
experiment_only_df.head()

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,tenure_check_correct,Variation,furthest_step_reached
0,836976,6.0,73.0,60.5,U,2.0,45105.30,6.0,9.0,6.0,Test,confirm
1,2304905,7.0,94.0,58.0,U,2.0,110860.30,6.0,9.0,7.0,Control,confirm
2,1439522,5.0,64.0,32.0,U,2.0,52467.79,6.0,9.0,5.0,Test,step_3
3,1562045,16.0,198.0,49.0,M,2.0,67454.65,3.0,6.0,16.0,Test,start
4,5126305,12.0,145.0,33.0,F,2.0,103671.75,0.0,3.0,12.0,Control,start


In [108]:
# Confirm the Test/Control split
experiment_only_df["Variation"].value_counts(dropna=False)

Variation
Test       26968
Control    23532
Name: count, dtype: int64

In [109]:
# Inspect rows with missing demographic data before dropping them
missing_rows = experiment_only_df[experiment_only_df.isnull().any(axis=1)]
missing_rows

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,tenure_check_correct,Variation,furthest_step_reached
4164,7402828,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Control,confirm
8316,355337,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Control,confirm
8677,8412164,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Test,step_2
9583,4666211,8.0,106.0,NaN,F,2.0,42550.55,4.0,7.0,8.0,Control,confirm
13444,2222915,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Test,step_1
18066,4876926,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Test,confirm
25961,5277910,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Test,confirm
28432,7616759,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Control,confirm
35323,8191345,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Control,confirm
43518,1227228,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Test,confirm


In [110]:
# Drop rows with missing values — this is the analysis-ready experiment dataset
experiment_final_df = experiment_only_df.dropna()

experiment_final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 50487 entries, 0 to 50499
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   client_id              50487 non-null  int64  
 1   clnt_tenure_yr         50487 non-null  float64
 2   clnt_tenure_mnth       50487 non-null  float64
 3   clnt_age               50487 non-null  float64
 4   gendr                  50487 non-null  object 
 5   num_accts              50487 non-null  float64
 6   bal                    50487 non-null  float64
 7   calls_6_mnth           50487 non-null  float64
 8   logons_6_mnth          50487 non-null  float64
 9   tenure_check_correct   50487 non-null  float64
 10  Variation              50487 non-null  object 
 11  furthest_step_reached  50487 non-null  object 
dtypes: float64(8), int64(1), object(3)
memory usage: 5.0+ MB


## Latest Client Process Step and Experiment Assignment

This analysis consolidates the web activity data to a client-level view by identifying the most recent interaction recorded for each client. The web activity dataset is first sorted chronologically, ensuring that the latest event for each client appears last in their activity history. The data is then grouped by `client_id`, and the final record for each client is retained, representing the furthest process step reached by that client.

The resulting dataset is enriched with experiment information by joining it with the experiment client dataset on `client_id`. This adds the `Variation` field, allowing each client to be classified into the corresponding experimental group (e.g., Test or Control).

As a validation step, individual client histories can be inspected by filtering the web activity dataset for a specific `client_id` and sorting the records by timestamp. This confirms that the process step retained in the final client-level dataset corresponds to the client's last recorded activity.

In [111]:
# Sort chronologically so each client's most recent event is last
combined_web_data_df = combined_web_data_df.sort_values("date_time")

# Get each client's latest recorded event
client_last_step = (
    combined_web_data_df
    .groupby("client_id")
    .last()
    .reset_index()
)

# Add each client's experiment group assignment
client_last_step = client_last_step.merge(
    experiment_clients_df[["client_id", "Variation"]],
    on="client_id",
    how="left"
)

client_last_step[["client_id", "process_step", "Variation"]].head(20)

,client_id,process_step,Variation
0,169,confirm,NaN
1,336,start,NaN
2,546,confirm,NaN
3,555,confirm,Test
4,647,confirm,Test
5,722,confirm,NaN
6,786,confirm,NaN
7,805,step_1,NaN
8,832,confirm,NaN
9,934,start,Test
